In [5]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from transformers import utils
utils.logging.set_verbosity_error()

from src.pipeline import PipeLine
from pathlib import Path
from src.translator import dataset_path

pipeline = PipeLine()
base_image_dir = dataset_path

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6529.99it/s]


## Match demonstration

In [6]:
text_query = "Look at this cute dog playing in the grass"
image_path = base_image_dir / "dog" / "OIP-_t431jQrfdq6YUVpvifqpQEsEs.jpeg"

print(f"Text: {text_query}")
match_result = pipeline.check_match(text_query, image_path)
print(f"Final match: {match_result}")

Text: Look at this cute dog playing in the grass
DEBUG Tokens: ['[CLS]', 'look', 'at', 'this', 'cute', 'dog', 'playing', 'in', 'the', 'grass', '[SEP]']
DEBUG Tags:   [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Found in text: dog
Found in image: dog
Final match: True


## Missmatch demonstration

In [11]:
text_query = "I see a big elephant"
image_path = base_image_dir / "cat" / "125.jpeg" # Image with cat

print(f"Text: {text_query}")
match_result = pipeline.check_match(text_query, image_path)
print(f"Final match: {match_result}")

Text: I see a big elephant
DEBUG Tokens: ['[CLS]', 'i', 'see', 'a', 'big', 'elephant', '[SEP]']
DEBUG Tags:   [0, 0, 0, 0, 0, 1, 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Found in text: elephant
Found in image: cat
Final match: False


# NER edge cases
* Few animals in request: algorithm will detect only first animal and ignore all other.
* Typos: Tokenizer will split a word with typo on unknown tokens and model will not detect target class.
* Context: If word used in another non-animal context model will mark it as an animal because it studied on simple sentences.

# CV edge cases
* Few animals on picture: model can't make object detection in current configuration, so it will choice only 1 animal class with the most powerful pattern.
* Unstandard draws: model learned on real life animals pictures only, 3D render or cartoon picture will drop down accuracy.
* No animal on the image: out-of-distribution case, model will output one of 10 classes of animals